## Selección de Modelo 

In [2]:
# Librerias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [ ]:
# Carga de datasets
movimientos = pd.read_csv("../dataset/movimientos.csv")
stock = pd.read_csv("../dataset/stock.csv")
productos = pd.read_csv("../dataset/productos.csv")

In [35]:
# Preprocesamiento del dataset
consumo_base = movimientos[movimientos['tipo_movimiento'] == 'S'].copy()
consumo_merged = consumo_base.merge(productos, on="gtin", how="left")

# Imputación de categorías nulas detectada en el EDA
consumo_merged['linea_terapeutica'] = consumo_merged['linea_terapeutica'].fillna('OTRO')
consumo_merged['uso_principal'] = consumo_merged['uso_principal'].fillna('OTRO')

# Seteo de variables temporales
consumo_merged['fecha_dt'] = pd.to_datetime(consumo_merged['fecha'])
consumo_merged['año'] = consumo_merged['fecha_dt'].dt.year
consumo_merged['mes'] = consumo_merged['fecha_dt'].dt.month
consumo_merged['dia_semana'] = consumo_merged['fecha_dt'].dt.dayofweek
consumo_merged['es_fin_de_semana'] = consumo_merged['dia_semana'].isin([5, 6]).astype(int)

# Estacionalidad cíclica mediante componentes armónicos (Seno/Coseno)
consumo_merged['mes_sin'] = np.sin(2 * np.pi * consumo_merged['mes'] / 12.0)
consumo_merged['mes_cos'] = np.cos(2 * np.pi * consumo_merged['mes'] / 12.0)

# Variable de tendencia
fecha_base = pd.to_datetime("2024-01-01")
consumo_merged['tendencia_dias'] = (consumo_merged['fecha_dt'] - fecha_base).dt.days

# División de la Muestra (Simulando el comportamiento de test_model.py)
X_source = consumo_merged.copy()
y_source = consumo_merged['cantidad']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_source, y_source, test_size=0.33, random_state=42
)

# Aplicación de target encoding de medias para linea terapeutica y uso principal
linea_mapeo = X_train_raw.groupby('linea_terapeutica')['cantidad'].mean().to_dict()
uso_mapeo = X_train_raw.groupby('uso_principal')['cantidad'].mean().to_dict()

# Mapear a entrenamiento y validación (asumiendo 0 para categorías no vistas)
X_train_raw['linea_encoded'] = X_train_raw['linea_terapeutica'].map(linea_mapeo).fillna(0)
X_train_raw['uso_encoded'] = X_train_raw['uso_principal'].map(uso_mapeo).fillna(0)

X_test_raw['linea_encoded'] = X_test_raw['linea_terapeutica'].map(linea_mapeo).fillna(0)
X_test_raw['uso_encoded'] = X_test_raw['uso_principal'].map(uso_mapeo).fillna(0)


In [36]:
# Selección de features y ajuste del modelo
features_cols = [
    'gtin', 'año', 'mes_sin', 'mes_cos',
    'tendencia_dias', 'dia_semana',
    'linea_encoded', 'uso_encoded'
]

X_train = X_train_raw[features_cols]
X_test = X_test_raw[features_cols]

# Entrenamiento del modelo y evaluación de métricas
modelo = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train.values.ravel())

# Predicción y cálculo del error
predicciones = modelo.predict(X_test)
mae_modelo = mean_absolute_error(y_test, predicciones)

In [37]:
# Baseline: predecir la media por producto del target de entrenamiento
train_means = pd.concat(
    [X_train[["gtin"]].reset_index(drop=True),
        y_train.reset_index(drop=True)], axis=1
)
product_means = train_means.groupby("gtin")["cantidad"].mean().to_dict()
global_mean = y_train.mean()
baseline_preds = [
    product_means.get(g, global_mean)
    for g in X_test["gtin"].values
]
mae_baseline = mean_absolute_error(y_test, baseline_preds)

In [39]:
# Comparativa contra el Benchmark exigido
porcentaje_error_baseline = (mae_modelo / mae_baseline) * 100
mejora_porcentual = 100 - porcentaje_error_baseline

print(f"MAE Baseline (Media por Producto): {mae_baseline:.4f}")
print(f"MAE Modelo (Random Forest):        {mae_modelo:.4f}")
print(f"Mejora respecto al benchmark:      {mejora_porcentual:.2f}%")

MAE Baseline (Media por Producto): 4.7828
MAE Modelo (Random Forest):        3.4401
Mejora respecto al benchmark:      28.08%
